# Aula 01: Algoritmos Gulosos (Greedy Algorithms)

No Módulo 01, focamos em métodos exatos que garantem a melhor solução possível. No entanto, para problemas complexos, o tempo de processamento pode ser inviável. Entramos agora no mundo das **Heurísticas**.

## 1. O que é um Algoritmo Guloso?

Um algoritmo guloso é qualquer algoritmo que segue a heurística de fazer a **escolha localmente ótima** em cada etapa, com a intenção de encontrar um ótimo global.

<div align="center">
  <img src="img/search_path.gif" width="400px" />
  <p><i>Exemplo: O algoritmo escolhe o maior valor disponível no próximo nível sem considerar o que vem depois.</i></p>
</div>

### Propriedades fundamentais:
Para que um algoritmo guloso retorne a solução ótima, o problema deve possuir:
1. **Propriedade da Escolha Gulosa**: Uma solução ótima global pode ser alcançada fazendo escolhas locais ótimas.
2. **Subestrutura Ótima**: Uma solução ótima para o problema contém em seu interior soluções ótimas para os sub-problemas.

## 2. Exemplo Clássico: O Problema do Troco (Coin Change)

Imagine que você precisa dar o troco de **36 centavos** usando o menor número de moedas possível.

<div align="center">
  <img src="img/coin_change.png" width="500px" />
</div>

**Estratégia Gulosa**: Sempre pegue a maior moeda possível que não ultrapasse o valor restante.
- Moedas disponíveis: 25, 10, 5, 1.
- 36 - **25** = 11
- 11 - **10** = 1
- 1 - **1** = 0
**Resultado**: 3 moedas (25, 10, 1). **Ótimo!**

In [ ]:
def dar_troco(valor, moedas):
    moedas = sorted(moedas, reverse=True)
    resultado = []
    for moeda in moedas:
        while valor >= moeda:
            valor -= moeda
            resultado.append(moeda)
    return resultado

moedas_br = [100, 50, 25, 10, 5, 1] # Real: Moedas em centavos
print(f"Troco de 87 centavos: {dar_troco(87, moedas_br)}")

## 3. TSP: Heurística do Vizinho Mais Próximo (Nearest Neighbor)

No TSP, o algoritmo guloso visita sempre a cidade mais próxima ainda não visitada.

### Algoritmo NN:
1. Comece em uma cidade inicial.
2. Encontre a cidade não visitada mais próxima e viaje para ela.
3. Repita até visitar todas.
4. Retorne à origem.

In [1]:
import math
import matplotlib.pyplot as plt
import json

def dist(c1, c2):
    return math.sqrt((c1['x'] - c2['x'])**2 + (c1['y'] - c2['y'])**2)

cidades = [
    {"id": 0, "x": 0, "y": 0}, {"id": 1, "x": 10, "y": 0},
    {"id": 2, "x": 10, "y": 10}, {"id": 3, "x": 0, "y": 10},
    {"id": 4, "x": 5, "y": 5}, {"id": 5, "x": 2, "y": 8},
    {"id": 6, "x": 8, "y": 2}
]

def solve_nn(lista_cidades, start_node=0):
    unvisited = lista_cidades.copy()
    current = unvisited.pop(start_node)
    tour = [current]
    total_dist = 0
    
    while unvisited:
        next_node = min(unvisited, key=lambda c: dist(current, c))
        total_dist += dist(current, next_node)
        tour.append(next_node)
        unvisited.remove(next_node)
        current = next_node
    
    total_dist += dist(tour[-1], tour[0])
    return tour, total_dist

tour_nn, d_nn = solve_nn(cidades)
print(f"Distância (Guloso): {d_nn:.2f}")

# Visualização
plt.figure(figsize=(8, 6))
xs = [c['x'] for c in tour_nn] + [tour_nn[0]['x']]
ys = [c['y'] for c in tour_nn] + [tour_nn[0]['y']]
plt.plot(xs, ys, 'o-', color='green', label='Rota Gulosa')
plt.title(f"TSP Guloso (Vizinho Mais Próximo)")
plt.grid(True)
plt.show()

Distância (Guloso): 45.22


## 4. Quando o Guloso Falha?

Um algoritmo guloso **não garante** a solução ótima para muitos problemas.

### O Contra-exemplo das Moedas
Se as moedas fossem de **1, 3 e 4** centavos, e precisássemos dar o troco de **6 centavos**:
- **Guloso**: 4 + 1 + 1 (3 moedas)
- **Ótimo**: 3 + 3 (2 moedas)

Aqui, a escolha local (o 4) impediu a melhor solução global.

## 5. O Problema da Mochila: Fracionária vs Inteira (0-1)

Um dos exemplos mais didáticos sobre a eficácia de algoritmos gulosos é a distinção entre a mochila fracionária e a mochila 0-1.

### 5.1 Mochila Fracionária (Fractional Knapsack)
Neste caso, você pode levar pedaços de um item (ex: quilos de arroz ou litros de óleo). 
**A estratégia gulosa é ÓTIMA**: Sempre pegue o item com a maior **densidade de valor** (Valor / Peso).

### 5.2 Mochila 0-1 (Zero-One Knapsack)
Aqui, você deve levar o item inteiro ou deixá-lo (ex: um notebook, uma TV). Não se pode levar "meio notebook".
**A estratégia gulosa falha**: Semelhante ao problema das moedas, a escolha gulosa pode ocupar um volume que impede a entrada de outros itens mais vantajosos em conjunto.

In [ ]:
def mochila_fracionaria(capacidade, itens):
    # 1. Ordenar por densidade de valor (V/W)
    itens_ordenados = sorted(itens, key=lambda x: x['valor']/x['peso'], reverse=True)
    
    valor_total = 0
    mochila = []
    
    for item in itens_ordenados:
        if capacidade <= 0:
            break
            
        if item['peso'] <= capacidade:
            # Leva o item inteiro
            capacidade -= item['peso']
            valor_total += item['valor']
            mochila.append((item['id'], "100%"))
        else:
            # Leva apenas uma fração do item
            fracao = capacidade / item['peso']
            valor_total += item['valor'] * fracao
            mochila.append((item['id'], f"{fracao*100:.1f}%"))
            capacidade = 0
            
    return mochila, valor_total

itens_mochila = [
    {'id': 'Ouro', 'valor': 500, 'peso': 10},
    {'id': 'Prata', 'valor': 200, 'peso': 20},
    {'id': 'Bronze', 'valor': 100, 'peso': 20}
]

res_itens, res_valor = mochila_fracionaria(25, itens_mochila)
print(f"Plano de Carga (Fracionário): {res_itens}")
print(f"Valor Total Alcançado: R$ {res_valor:.2f}")

## 6. Conclusão da Aula

Nesta aula, vimos que algoritmos gulosos são ferramentas poderosas devido à sua **simplicidade e velocidade**. 

- Eles são ideais quando precisamos de uma solução "boa o suficiente" em milissegundos.
- Eles servem como excelente base (soluções iniciais) para metaheurísticas mais complexas.
- **Lembre-se**: Sempre verifique se o seu problema possui **subestrutura ótima** e a **propriedade da escolha gulosa** antes de confiar plenamente no resultado.

Na próxima aula, exploraremos a **Busca Local**, uma técnica que tenta melhorar soluções (como esta que geramos para o TSP) através de pequenas modificações iterativas.